# Distributed Renewable Integration Stability Forecasting
## End-to-End Pipeline Notebook

**Project:** Predicting grid instability from 10-year hourly NASA POWER weather data for Northern Ireland  
**Dataset:** 2015 – 2025 | ~87,660 hourly records | 11 meteorological variables  

### Pipeline Overview
| Section | Module | What It Does |
|---------|--------|--------------|
| 1 | `config/spark_config.py` | Spark session + path constants |
| 2 | `src/data_ingestion.py` | Load CSV → Spark → HDFS-sim Parquet |
| 3 | `src/data_preprocessing.py` | Handle -999, datetime index, validate |
| 4 | `src/eda.py` | 12 diagnostic plots (distributions, correlations, decomposition) |
| 5 | `src/feature_engineering.py` | WPI, SII, temporal, interaction, lags, rolling, RVSI, risk labels |
| 6 | `src/modeling.py` | Random Forest + GBT (regression & classification) via PySpark MLlib |
| 7 | `src/time_series.py` | SARIMA(1,1,1)(1,1,1,7) daily RVSI forecasting |
| 8 | `src/deep_learning.py` | LSTM regressor + DNN classifier (TensorFlow/Keras) |
| 9 | — | Results summary & model comparison |


## Section 1: Setup & Configuration

In [2]:
import os
import sys
from pathlib import Path

# Ensure project root is on sys.path so src.* imports work
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT)) 
print(f"Project root: {PROJECT_ROOT}")

# Suppress TF info messages (optional)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless backend – saves PNGs instead of popping windows
import matplotlib.pyplot as plt
import seaborn as sns

print("Core imports OK")

Project root: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact
Core imports OK


In [3]:
from config.spark_config import (
    get_spark_session,
    PROJECT_ROOT   as CFG_ROOT,
    DATA_DIR, RAW_DATA_DIR, CLEAN_DATA_DIR,
    HDFS_SIM_DIR, MODEL_DIR,
    KAGGLE_DATASET, CSV_FILENAME,
    MISSING_VALUE_SENTINEL,
)

print("Config loaded:")
print(f"  DATA_DIR      : {DATA_DIR}")
print(f"  RAW_DATA_DIR  : {RAW_DATA_DIR}")
print(f"  CLEAN_DATA_DIR: {CLEAN_DATA_DIR}")
print(f"  HDFS_SIM_DIR  : {HDFS_SIM_DIR}")
print(f"  MODEL_DIR     : {MODEL_DIR}")
print(f"  MISSING SENTINEL: {MISSING_VALUE_SENTINEL}")

Config loaded:
  DATA_DIR      : /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/data
  RAW_DATA_DIR  : /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/data/raw
  CLEAN_DATA_DIR: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/data/clean
  HDFS_SIM_DIR  : /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/hdfs_sim
  MODEL_DIR     : /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/models
  MISSING SENTINEL: -999


In [4]:
# Initialise Spark (local mode – simulates distributed execution)
spark = get_spark_session()
print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 16:09:41 WARN Utils: Your hostname, LAPTOP-Q1GG77AG, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/04 16:09:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 16:09:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.1.1
App name      : RenewableStabilityForecasting
Master        : local[*]


## Section 2: Data Ingestion & HDFS Simulation

In [5]:
from src.data_ingestion import download_dataset, load_raw_csv, partition_to_hdfs_sim

# Step 2a: Download / locate raw CSV
csv_path = download_dataset()
print(f"CSV path: {csv_path}")

[OK] Dataset already exists: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/data/raw/nasa_power_2015to2025_hourly_data_NORTHERN_IRELAND.csv
CSV path: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/data/raw/nasa_power_2015to2025_hourly_data_NORTHERN_IRELAND.csv


In [6]:
# Step 2b: Load raw CSV into a Spark DataFrame
df_raw = load_raw_csv(spark)
print(f"Raw rows: {df_raw.count():,}  |  Columns: {len(df_raw.columns)}")
print("\nSchema:")
df_raw.printSchema()


------------------------------------------------------------
  Loading Raw CSV into Spark DataFrame
------------------------------------------------------------




------------------------------------------------------------
  Raw NASA POWER Data Summary
------------------------------------------------------------

  Rows:    89,376
  Columns: 15

  Schema:
root
 |-- YEAR: integer (nullable = true)
 |-- MO: integer (nullable = true)
 |-- DY: integer (nullable = true)
 |-- HR: integer (nullable = true)
 |-- ALLSKY_SFC_SW_DWN: double (nullable = true)
 |-- T2M: double (nullable = true)
 |-- RH2M: double (nullable = true)
 |-- PS: double (nullable = true)
 |-- WS10M: double (nullable = true)
 |-- WD10M: double (nullable = true)
 |-- WS50M: double (nullable = true)
 |-- WD50M: double (nullable = true)
 |-- RHOA: double (nullable = true)
 |-- QV10M: double (nullable = true)
 |-- CLOUD_AMT: double (nullable = true)


  First 5 rows:
+----+---+---+---+-----------------+----+-----+------+-----+-----+-----+-----+----+-----+---------+
|YEAR|MO |DY |HR |ALLSKY_SFC_SW_DWN|T2M |RH2M |PS    |WS10M|WD10M|WS50M|WD50M|RHOA|QV10M|CLOUD_AMT|
+----+---+---+---+----

In [7]:
# Step 2c: Quick look at the raw data
df_raw.show(5, truncate=False)

+----+---+---+---+-----------------+----+-----+------+-----+-----+-----+-----+----+-----+---------+
|YEAR|MO |DY |HR |ALLSKY_SFC_SW_DWN|T2M |RH2M |PS    |WS10M|WD10M|WS50M|WD50M|RHOA|QV10M|CLOUD_AMT|
+----+---+---+---+-----------------+----+-----+------+-----+-----+-----+-----+----+-----+---------+
|2015|1  |1  |0  |0.0              |8.14|96.55|100.49|7.92 |212.9|11.17|214.3|1.24|6.5  |99.68    |
|2015|1  |1  |1  |0.0              |8.21|96.56|100.36|8.16 |204.9|11.32|206.3|1.24|6.54 |99.97    |
|2015|1  |1  |2  |0.0              |8.34|96.58|100.35|7.83 |207.4|11.04|208.6|1.24|6.6  |100.0    |
|2015|1  |1  |3  |0.0              |8.29|95.62|100.38|7.91 |213.3|11.26|214.5|1.24|6.49 |100.0    |
|2015|1  |1  |4  |0.0              |7.99|96.28|100.26|6.91 |207.2|9.97 |208.4|1.24|6.42 |99.69    |
+----+---+---+---+-----------------+----+-----+------+-----+-----+-----+-----+----+-----+---------+
only showing top 5 rows


In [8]:
# Step 2d: Partition data into HDFS-simulated Parquet (partitioned by YEAR)
partition_to_hdfs_sim(df_raw)
print("HDFS simulation partitioning complete.")


------------------------------------------------------------
  Writing to Simulated HDFS (Partitioned Parquet)
------------------------------------------------------------

  Output: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/hdfs_sim/nasa_power_partitioned
  Partition column: YEAR


  [OK] Created 11 partitions: ['YEAR=2015', 'YEAR=2016', 'YEAR=2017', 'YEAR=2018', 'YEAR=2019', 'YEAR=2020', 'YEAR=2021', 'YEAR=2022', 'YEAR=2023', 'YEAR=2024', 'YEAR=2025']
HDFS simulation partitioning complete.


## Section 3: Data Preprocessing

In [9]:
from src.data_preprocessing import (
    handle_missing_values,
    create_datetime_column,
    validate_schema,
    clean_pipeline,
)

# Step 3a: Handle sentinel missing values (-999 -> null)
df_nomiss, n_replaced = handle_missing_values(df_raw)
print(f"Total sentinel values replaced: {n_replaced:,}")


------------------------------------------------------------
  Handling Missing Values (Sentinel: -999)
------------------------------------------------------------

  ALLSKY_SFC_SW_DWN: 5,400 sentinel values found
  T2M: 96 sentinel values found
  RH2M: 96 sentinel values found
  PS: 96 sentinel values found
  WS10M: 96 sentinel values found
  WD10M: 96 sentinel values found
  WS50M: 96 sentinel values found
  WD50M: 96 sentinel values found
  RHOA: 96 sentinel values found
  QV10M: 96 sentinel values found
  CLOUD_AMT: 5,400 sentinel values found

  Total sentinel values to replace: 11,664

  After replacement -- null counts per column:
    ALLSKY_SFC_SW_DWN: 5,400 nulls
    T2M: 96 nulls
    RH2M: 96 nulls
    PS: 96 nulls
    WS10M: 96 nulls
    WD10M: 96 nulls
    WS50M: 96 nulls
    WD50M: 96 nulls
    RHOA: 96 nulls


    QV10M: 96 nulls
    CLOUD_AMT: 5,400 nulls

  [OK] Replaced 11,664 sentinel values with null
Total sentinel values replaced: 11,664


In [10]:
# Step 3b: Create unified timestamp column from YEAR / MO / DY / HR
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_ts = create_datetime_column(df_nomiss)
df_ts.select("YEAR", "MO", "DY", "HR", "timestamp").show(5)


------------------------------------------------------------
  Creating Datetime Column
------------------------------------------------------------

  [OK] All timestamps created successfully
  Date range: 2015-01-01 00:00:00 -> 2025-03-12 23:00:00
  [OK] DataFrame sorted by timestamp
+----+---+---+---+-------------------+
|YEAR| MO| DY| HR|          timestamp|
+----+---+---+---+-------------------+
|2015|  1|  1|  0|2015-01-01 00:00:00|
|2015|  1|  1|  1|2015-01-01 01:00:00|
|2015|  1|  1|  2|2015-01-01 02:00:00|
|2015|  1|  1|  3|2015-01-01 03:00:00|
|2015|  1|  1|  4|2015-01-01 04:00:00|
+----+---+---+---+-------------------+
only showing top 5 rows


In [11]:
# Step 3c: Validate schema (required columns present, row count reasonable)
is_valid = validate_schema(df_ts)
print(f"Schema valid: {is_valid}")


------------------------------------------------------------
  Validating Schema
------------------------------------------------------------

  [OK] All 16 required columns present
  [OK] Total columns: 16
  [OK] Row count: 89,376
Schema valid: True


In [12]:
# Step 3d: Run full clean pipeline (handles missing + datetime + validation + writes Parquet)
df_clean = clean_pipeline(spark)
print(f"Cleaned rows   : {df_clean.count():,}")
print(f"Cleaned columns: {len(df_clean.columns)}")
df_clean.show(3)


  DATA PREPROCESSING PIPELINE


------------------------------------------------------------
  Step 1: Loading Raw Data
------------------------------------------------------------



  Loaded 89,376 rows, 15 columns

------------------------------------------------------------
  Handling Missing Values (Sentinel: -999)
------------------------------------------------------------

  ALLSKY_SFC_SW_DWN: 5,400 sentinel values found
  T2M: 96 sentinel values found
  RH2M: 96 sentinel values found
  PS: 96 sentinel values found
  WS10M: 96 sentinel values found
  WD10M: 96 sentinel values found
  WS50M: 96 sentinel values found
  WD50M: 96 sentinel values found
  RHOA: 96 sentinel values found
  QV10M: 96 sentinel values found
  CLOUD_AMT: 5,400 sentinel values found

  Total sentinel values to replace: 11,664

  After replacement -- null counts per column:
    ALLSKY_SFC_SW_DWN: 5,400 nulls
    T2M: 96 nulls
    RH2M: 96 nulls
    PS: 96 nulls
    WS10M: 96 nulls
    WD10M: 96 nulls
    WS50M: 96 nulls
    WD50M: 96 nulls
    RHOA: 96 nulls
    QV10M: 96 nulls
    CLOUD_AMT: 5,400 nulls

  [OK] Replaced 11,664 sentinel values with null

---------------------------------


  [OK] Wrote 89,376 rows to Parquet
  [OK] Sentinel values handled: 11,664

  DATA PREPROCESSING COMPLETE

Cleaned rows   : 89,376
Cleaned columns: 16
+----+---+---+---+-----------------+----+-----+------+-----+-----+-----+-----+----+-----+---------+-------------------+
|YEAR| MO| DY| HR|ALLSKY_SFC_SW_DWN| T2M| RH2M|    PS|WS10M|WD10M|WS50M|WD50M|RHOA|QV10M|CLOUD_AMT|          timestamp|
+----+---+---+---+-----------------+----+-----+------+-----+-----+-----+-----+----+-----+---------+-------------------+
|2015|  1|  1|  0|              0.0|8.14|96.55|100.49| 7.92|212.9|11.17|214.3|1.24|  6.5|    99.68|2015-01-01 00:00:00|
|2015|  1|  1|  1|              0.0|8.21|96.56|100.36| 8.16|204.9|11.32|206.3|1.24| 6.54|    99.97|2015-01-01 01:00:00|
|2015|  1|  1|  2|              0.0|8.34|96.58|100.35| 7.83|207.4|11.04|208.6|1.24|  6.6|    100.0|2015-01-01 02:00:00|
+----+---+---+---+-----------------+----+-----+------+-----+-----+-----+-----+----+-----+---------+-------------------+
only sho


## Section 4: Exploratory Data Analysis (EDA)

Runs 12 diagnostic plots — all saved as PNGs to `data/plots/`.

In [13]:
from src.eda import (
    plot_descriptive_stats,
    plot_distributions,
    plot_correlation_heatmap,
    plot_time_series_trends,
    plot_hourly_patterns,
    plot_seasonal_boxplots,
    plot_wind_direction,
    plot_missing_data,
    plot_seasonal_decomposition,
    plot_acf_pacf,
    plot_rvsi_analysis,
    plot_wpi_vs_sii,
    run_eda,
)

PLOT_DIR = PROJECT_ROOT / "data" / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plot output directory: {PLOT_DIR}")

Plot output directory: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/data/plots


In [14]:
# Convert clean Spark DF to Pandas for EDA (EDA operates on Pandas)
pdf_clean = df_clean.toPandas()
pdf_clean["timestamp"] = pd.to_datetime(pdf_clean["timestamp"])
print(f"Pandas shape: {pdf_clean.shape}")
pdf_clean.head(3)

Pandas shape: (89376, 16)


,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DWN,T2M,RH2M,PS,WS10M,WD10M,WS50M,WD50M,RHOA,QV10M,CLOUD_AMT,timestamp
0,2015,1,1,0,0.0,8.14,96.55,100.49,7.92,212.9,11.17,214.3,1.24,6.50,99.68,2015-01-01 00:00:00
1,2015,1,1,1,0.0,8.21,96.56,100.36,8.16,204.9,11.32,206.3,1.24,6.54,99.97,2015-01-01 01:00:00
2,2015,1,1,2,0.0,8.34,96.58,100.35,7.83,207.4,11.04,208.6,1.24,6.60,100.00,2015-01-01 02:00:00


In [15]:
# Plot 1: Descriptive Statistics
plot_descriptive_stats(pdf_clean)
print("\nDescriptive stats:")
pdf_clean.describe().round(2)

  Saved: 01_descriptive_statistics.png

Descriptive stats:


,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DWN,T2M,RH2M,PS,WS10M,WD10M,WS50M,WD50M,RHOA,QV10M,CLOUD_AMT,timestamp
count,89376.00,89376.00,89376.00,89376.00,83976.00,89280.00,89280.00,89280.00,89280.00,89280.00,89280.00,89280.00,89280.00,89280.00,83976.00,89376
mean,2019.61,6.43,15.69,11.50,109.02,9.00,89.87,99.91,5.21,208.18,7.48,208.70,1.23,6.53,76.96,2020-02-05 23:30:00.402792960
min,2015.00,1.00,1.00,0.00,0.00,-3.50,43.27,94.08,0.01,0.00,0.01,0.00,1.15,2.25,0.00,2015-01-01 00:00:00
25%,2017.00,3.00,8.00,5.75,0.00,5.15,85.98,99.13,3.15,151.40,5.21,152.40,1.21,5.04,61.15,2017-07-19 23:45:00
50%,2020.00,6.00,16.00,11.50,6.43,8.80,93.93,100.04,4.79,219.40,7.14,220.40,1.23,6.32,91.29,2020-02-05 23:30:00
75%,2022.00,9.00,23.00,17.25,169.75,12.65,96.85,100.79,6.76,272.42,9.25,272.90,1.25,7.77,99.38,2022-08-24 23:15:00
max,2025.00,12.00,31.00,23.00,885.90,27.25,100.00,103.57,26.81,359.90,34.27,360.00,1.32,15.14,100.00,2025-03-12 23:00:00
std,2.94,3.48,8.80,6.92,167.09,5.09,10.00,1.27,2.65,86.85,3.34,86.86,0.03,1.91,28.55,NaN


In [16]:
# Plot 2: Distributions (Histograms + KDE)
plot_distributions(pdf_clean)
img = plt.imread(str(PLOT_DIR / "02_distributions.png"))
plt.figure(figsize=(18, 7)); plt.imshow(img); plt.axis("off")
plt.title("Fig 2: Distributions of Meteorological Variables"); plt.tight_layout(); plt.show()

  Saved: 02_distributions.png


In [17]:
# Plot 3: Correlation Heatmap
plot_correlation_heatmap(pdf_clean)
img = plt.imread(str(PLOT_DIR / "03_correlation_heatmap.png"))
plt.figure(figsize=(12, 10)); plt.imshow(img); plt.axis("off")
plt.title("Fig 3: Pearson Correlation Matrix"); plt.tight_layout(); plt.show()

  Saved: 03_correlation_heatmap.png


In [18]:
# Plot 4: 10-Year Monthly Trend (Wind / Solar / Temperature)
plot_time_series_trends(pdf_clean)
img = plt.imread(str(PLOT_DIR / "04_time_series_trends.png"))
plt.figure(figsize=(16, 10)); plt.imshow(img); plt.axis("off")
plt.title("Fig 4: 10-Year Monthly Average Trends"); plt.tight_layout(); plt.show()

  Saved: 04_time_series_trends.png


In [19]:
# Plot 5: Diurnal Patterns (Hour of Day)
plot_hourly_patterns(pdf_clean)
img = plt.imread(str(PLOT_DIR / "05_hourly_patterns.png"))
plt.figure(figsize=(14, 5)); plt.imshow(img); plt.axis("off")
plt.title("Fig 5: Diurnal Patterns"); plt.tight_layout(); plt.show()

  Saved: 05_hourly_patterns.png


In [20]:
# Plot 6: Seasonal Box Plots
plot_seasonal_boxplots(pdf_clean)
img = plt.imread(str(PLOT_DIR / "06_seasonal_boxplots.png"))
plt.figure(figsize=(20, 5)); plt.imshow(img); plt.axis("off")
plt.title("Fig 6: Seasonal Box Plots"); plt.tight_layout(); plt.show()

  Saved: 06_seasonal_boxplots.png


In [21]:
# Plot 7: Wind Direction Histogram
plot_wind_direction(pdf_clean)
img = plt.imread(str(PLOT_DIR / "07_wind_direction.png"))
plt.figure(figsize=(14, 5)); plt.imshow(img); plt.axis("off")
plt.title("Fig 7: Wind Direction Distribution"); plt.tight_layout(); plt.show()

  Saved: 07_wind_direction.png


In [22]:
# Plot 8: Missing Data (Sentinel -999) Heatmap — uses raw CSV
raw_csv_files = list(RAW_DATA_DIR.glob("*.csv"))
if raw_csv_files:
    pdf_raw_check = pd.read_csv(str(raw_csv_files[0]))
    plot_missing_data(pdf_raw_check)
    img = plt.imread(str(PLOT_DIR / "08_missing_data_heatmap.png"))
    plt.figure(figsize=(14, 6)); plt.imshow(img); plt.axis("off")
    plt.title("Fig 8: Sentinel Value Distribution"); plt.tight_layout(); plt.show()
else:
    print("Raw CSV not found – skipping plot 8.")

  Saved: 08_missing_data_heatmap.png


In [23]:
# Plot 9: Seasonal Decomposition (Solar & Wind — daily aggregates)
plot_seasonal_decomposition(pdf_clean)
img = plt.imread(str(PLOT_DIR / "09_seasonal_decomposition.png"))
plt.figure(figsize=(18, 14)); plt.imshow(img); plt.axis("off")
plt.title("Fig 9: Seasonal Decomposition (Trend / Season / Residual)"); plt.tight_layout(); plt.show()

  Saved: 09_seasonal_decomposition.png


In [24]:
# Plot 10: ACF / PACF (Autocorrelation)
plot_acf_pacf(pdf_clean)
img = plt.imread(str(PLOT_DIR / "10_acf_pacf.png"))
plt.figure(figsize=(16, 8)); plt.imshow(img); plt.axis("off")
plt.title("Fig 10: Autocorrelation (ACF / PACF)"); plt.tight_layout(); plt.show()

  Saved: 10_acf_pacf.png



## Section 5: Feature Engineering

All feature engineering runs **in Spark** (distributed).  
New columns added:

| Group | Columns |
|---|---|
| Energy Indices | `wind_power_index`, `solar_irradiance_index` |
| Temporal | `hour`, `month`, `season`, `is_daytime`, `hour_sin/cos`, `month_sin/cos` |
| Interaction | `renewable_combined_index`, `temp_humidity_index`, `wind_solar_ratio`, `pressure_density_index`, `wind_direction_variability`, `wind_speed_ratio` |
| Lag Features | `*_lag_{1,6,12,24}h` (4 columns × 4 lags = 16 cols) |
| Rolling Stats | `*_rolling_{mean,std}_{6,12,24}h` (2 cols × 2 stats × 3 windows = 12 cols) |
| Volatility | `wind_cv_24h`, `solar_cv_24h`, `rvsi` |
| Risk Label | `instability_risk` (Low / Medium / High / Critical) |

In [25]:
from src.feature_engineering import (
    compute_wind_power_index,
    compute_solar_irradiance_index,
    add_temporal_features,
    add_interaction_features,
    add_lag_features,
    add_rolling_stats,
    compute_rvsi,
    label_instability_risk,
    feature_pipeline,
)

print("Feature engineering functions imported.")

Feature engineering functions imported.


In [26]:
# Step 5a: Wind Power Index  WPI = 0.5 × ρ × v³
df_feat = compute_wind_power_index(df_clean)
df_feat.select("timestamp", "RHOA", "WS50M", "wind_power_index").show(5)


------------------------------------------------------------
  Computing Wind Power Index (WPI)
------------------------------------------------------------

  WPI Stats -- Mean: 426.44, Std: 649.68
              Min:  0.00, Max: 23746.21
  [OK] wind_power_index column added
+-------------------+----+-----+-----------------+
|          timestamp|RHOA|WS50M| wind_power_index|
+-------------------+----+-----+-----------------+
|2015-01-01 00:00:00|1.24|11.17|864.0745400599999|
|2015-01-01 01:00:00|1.24|11.32|899.3546201600001|
|2015-01-01 02:00:00|1.24|11.04|834.2551756799999|
|2015-01-01 03:00:00|1.24|11.26|     885.12959312|
|2015-01-01 04:00:00|1.24| 9.97|614.4367232600001|
+-------------------+----+-----+-----------------+
only showing top 5 rows


In [27]:
# Step 5b: Solar Irradiance Index  SII = Irradiance × (1 - Cloud/100) × temp_factor
df_feat = compute_solar_irradiance_index(df_feat)
df_feat.select("timestamp", "ALLSKY_SFC_SW_DWN", "CLOUD_AMT", "T2M", "solar_irradiance_index").show(5)


------------------------------------------------------------
  Computing Solar Irradiance Index (SII)
------------------------------------------------------------

  SII Stats -- Mean: 30.58, Max: 870.49
  [OK] solar_irradiance_index column added
+-------------------+-----------------+---------+----+----------------------+
|          timestamp|ALLSKY_SFC_SW_DWN|CLOUD_AMT| T2M|solar_irradiance_index|
+-------------------+-----------------+---------+----+----------------------+
|2015-01-01 00:00:00|              0.0|    99.68|8.14|                   0.0|
|2015-01-01 01:00:00|              0.0|    99.97|8.21|                   0.0|
|2015-01-01 02:00:00|              0.0|    100.0|8.34|                   0.0|
|2015-01-01 03:00:00|              0.0|    100.0|8.29|                   0.0|
|2015-01-01 04:00:00|              0.0|    99.69|7.99|                   0.0|
+-------------------+-----------------+---------+----+----------------------+
only showing top 5 rows


In [28]:
# Step 5c: Temporal Features (hour, month, season, is_daytime, cyclical encoding)
df_feat = add_temporal_features(df_feat)
df_feat.select("timestamp", "hour", "month", "season", "is_daytime",
               "hour_sin", "hour_cos", "month_sin", "month_cos").show(5)


------------------------------------------------------------
  Adding Temporal Features
------------------------------------------------------------

  [OK] Added 8 temporal features (hour, month, season, is_daytime, cyclical)
+-------------------+----+-----+------+----------+-------------------+------------------+-------------------+------------------+
|          timestamp|hour|month|season|is_daytime|           hour_sin|          hour_cos|          month_sin|         month_cos|
+-------------------+----+-----+------+----------+-------------------+------------------+-------------------+------------------+
|2015-01-01 00:00:00|   0|    1|     0|         0|                0.0|               1.0|0.49999999999999994|0.8660254037844387|
|2015-01-01 01:00:00|   1|    1|     0|         0|0.25881904510252074|0.9659258262890683|0.49999999999999994|0.8660254037844387|
|2015-01-01 02:00:00|   2|    1|     0|         0|0.49999999999999994|0.8660254037844387|0.49999999999999994|0.8660254037844387

In [29]:
# Step 5d: Interaction Features (cross-variable combinations)
df_feat = add_interaction_features(df_feat)
df_feat.select(
    "timestamp",
    "renewable_combined_index",
    "temp_humidity_index",
    "wind_solar_ratio",
    "pressure_density_index",
    "wind_direction_variability",
    "wind_speed_ratio",
).show(5)


------------------------------------------------------------
  Adding Interaction Features
------------------------------------------------------------

  [OK] Added 6 interaction features
+-------------------+------------------------+-------------------+-----------------+----------------------+--------------------------+------------------+
|          timestamp|renewable_combined_index|temp_humidity_index| wind_solar_ratio|pressure_density_index|wind_direction_variability|  wind_speed_ratio|
+-------------------+------------------------+-------------------+-----------------+----------------------+--------------------------+------------------+
|2015-01-01 00:00:00|       864.0745400599999|  7.859170000000001|864.0745400599999|    124.60759999999999|        1.4000000000000057|1.3927680798004989|
|2015-01-01 01:00:00|       899.3546201600001|  7.927576000000001|899.3546201600001|              124.4464|        1.4000000000000057|1.3704600484261502|
|2015-01-01 02:00:00|       834.25517567

In [30]:
# Step 5e: Lag Features (1h, 6h, 12h, 24h lags for WPI, SII, WS50M, Solar)
df_feat = add_lag_features(df_feat)
lag_cols = [c for c in df_feat.columns if "_lag_" in c]
print(f"Lag columns added: {len(lag_cols)}")
print(lag_cols[:8])
df_feat.select(["timestamp"] + lag_cols[:4]).show(5)


------------------------------------------------------------
  Adding Lag Features
------------------------------------------------------------

  [OK] Added 16 lag features (4 columns x 4 lags)
Lag columns added: 16
['wind_power_index_lag_1h', 'wind_power_index_lag_6h', 'wind_power_index_lag_12h', 'wind_power_index_lag_24h', 'solar_irradiance_index_lag_1h', 'solar_irradiance_index_lag_6h', 'solar_irradiance_index_lag_12h', 'solar_irradiance_index_lag_24h']


26/05/04 16:11:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

+-------------------+-----------------------+-----------------------+------------------------+------------------------+
|          timestamp|wind_power_index_lag_1h|wind_power_index_lag_6h|wind_power_index_lag_12h|wind_power_index_lag_24h|
+-------------------+-----------------------+-----------------------+------------------------+------------------------+
|2015-01-01 00:00:00|                   NULL|                   NULL|                    NULL|                    NULL|
|2015-01-01 01:00:00|      864.0745400599999|                   NULL|                    NULL|                    NULL|
|2015-01-01 02:00:00|      899.3546201600001|                   NULL|                    NULL|                    NULL|
|2015-01-01 03:00:00|      834.2551756799999|                   NULL|                    NULL|                    NULL|
|2015-01-01 04:00:00|           885.12959312|                   NULL|                    NULL|                    NULL|
+-------------------+-------------------

In [31]:
# Step 5f: Rolling Statistics (6h, 12h, 24h mean + std for WPI and SII)
df_feat = add_rolling_stats(df_feat)
rolling_cols = [c for c in df_feat.columns if "_rolling_" in c]
print(f"Rolling stat columns added: {len(rolling_cols)}")
print(rolling_cols)
df_feat.select(["timestamp"] + rolling_cols[:4]).show(5)


------------------------------------------------------------
  Adding Rolling Statistics
------------------------------------------------------------

  [OK] Added 12 rolling features (2 cols x 3 windows x 2 stats)
Rolling stat columns added: 12
['wind_power_index_rolling_mean_6h', 'wind_power_index_rolling_std_6h', 'wind_power_index_rolling_mean_12h', 'wind_power_index_rolling_std_12h', 'wind_power_index_rolling_mean_24h', 'wind_power_index_rolling_std_24h', 'solar_irradiance_index_rolling_mean_6h', 'solar_irradiance_index_rolling_std_6h', 'solar_irradiance_index_rolling_mean_12h', 'solar_irradiance_index_rolling_std_12h', 'solar_irradiance_index_rolling_mean_24h', 'solar_irradiance_index_rolling_std_24h']


26/05/04 16:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

+-------------------+--------------------------------+-------------------------------+---------------------------------+--------------------------------+
|          timestamp|wind_power_index_rolling_mean_6h|wind_power_index_rolling_std_6h|wind_power_index_rolling_mean_12h|wind_power_index_rolling_std_12h|
+-------------------+--------------------------------+-------------------------------+---------------------------------+--------------------------------+
|2015-01-01 00:00:00|               864.0745400599999|                           NULL|                864.0745400599999|                            NULL|
|2015-01-01 01:00:00|                    881.71458011|             24.946783879514705|                     881.71458011|              24.946783879514705|
|2015-01-01 02:00:00|               865.8947786333333|              32.58787150534762|                865.8947786333333|               32.58787150534762|
|2015-01-01 03:00:00|               870.7034822549999|             28.292651

In [32]:
# Step 5g: RVSI — Renewable Volatility Stress Index
# RVSI = 0.6 × (σ_wind_24h / μ_wind_24h) + 0.4 × (σ_solar_24h / μ_solar_24h)
df_feat = compute_rvsi(df_feat)
df_feat.select("timestamp", "wind_cv_24h", "solar_cv_24h", "rvsi").show(5)


------------------------------------------------------------
  Computing Renewable Volatility Stress Index (RVSI)
------------------------------------------------------------



26/05/04 16:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  RVSI Stats -- Mean: 1.1018, Std: 0.3729, P95: 1.7413
  [OK] rvsi column added
+-------------------+--------------------+------------+--------------------+
|          timestamp|         wind_cv_24h|solar_cv_24h|                rvsi|
+-------------------+--------------------+------------+--------------------+
|2015-01-01 00:00:00|                NULL|         0.0|                NULL|
|2015-01-01 01:00:00|0.028293491388565243|         0.0|0.016976094833139145|
|2015-01-01 02:00:00|0.037634909355593986|         0.0| 0.02258094561335639|
|2015-01-01 03:00:00| 0.03249401385643793|         0.0| 0.01949640831386276|
|2015-01-01 04:00:00|  0.1430177692409611|         0.0| 0.08581066154457666|
+-------------------+--------------------+------------+--------------------+
only showing top 5 rows


26/05/04 16:11:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [33]:
# Step 5h: Risk Labels (quantile-based: Low / Medium / High / Critical)
df_feat = label_instability_risk(df_feat)
df_feat.select("timestamp", "rvsi", "instability_risk").show(10)

# Write the feature-engineered dataset to Parquet
feat_output = str(CLEAN_DATA_DIR / "nasa_power_features")
df_feat.write.mode("overwrite").parquet(feat_output)
print(f"\nFeature data written to: {feat_output}")
print(f"Total columns: {len(df_feat.columns)}")


------------------------------------------------------------
  Labeling Instability Risk Classes
------------------------------------------------------------



26/05/04 16:11:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


  Thresholds -- Q25: 0.8737, Q50: 1.0786, Q90: 1.5497

  Risk distribution:


26/05/04 16:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------------+-----+
|instability_risk|count|
+----------------+-----+
|        Critical| 9567|
|            High|35175|
|             Low|21788|
|          Medium|22846|
+----------------+-----+

  [OK] instability_risk column added


26/05/04 16:11:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

+-------------------+--------------------+----------------+
|          timestamp|                rvsi|instability_risk|
+-------------------+--------------------+----------------+
|2015-01-01 00:00:00|                NULL|        Critical|
|2015-01-01 01:00:00|0.016976094833139145|             Low|
|2015-01-01 02:00:00| 0.02258094561335639|             Low|
|2015-01-01 03:00:00| 0.01949640831386276|             Low|
|2015-01-01 04:00:00| 0.08581066154457666|             Low|
|2015-01-01 05:00:00|  0.1089148936193033|             Low|
|2015-01-01 06:00:00|  0.1015452852195962|             Low|
|2015-01-01 07:00:00| 0.14437648522976101|             Low|
|2015-01-01 08:00:00|  0.3188615243730351|             Low|
|2015-01-01 09:00:00|  0.4790288500560863|             Low|
+-------------------+--------------------+----------------+
only showing top 10 rows


26/05/04 16:11:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/04 16:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance


Feature data written to: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/data/clean/nasa_power_features
Total columns: 64


In [34]:
# Plot 11: RVSI Distribution + Risk Class Pie
pdf_feat = df_feat.toPandas()
pdf_feat["timestamp"] = pd.to_datetime(pdf_feat["timestamp"])

plot_rvsi_analysis(pdf_feat)
img = plt.imread(str(PLOT_DIR / "11_rvsi_analysis.png"))
plt.figure(figsize=(14, 5)); plt.imshow(img); plt.axis("off")
plt.title("Fig 11: RVSI Distribution + Risk Class Proportion"); plt.tight_layout(); plt.show()

26/05/04 16:11:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  Saved: 11_rvsi_analysis.png


In [35]:
# Plot 12: WPI vs SII coloured by Risk Class
plot_wpi_vs_sii(pdf_feat)
img = plt.imread(str(PLOT_DIR / "12_wpi_vs_sii_scatter.png"))
plt.figure(figsize=(10, 7)); plt.imshow(img); plt.axis("off")
plt.title("Fig 12: WPI vs SII by Instability Risk"); plt.tight_layout(); plt.show()

  Saved: 12_wpi_vs_sii_scatter.png



## Section 6: Machine Learning Modeling (PySpark MLlib)

Models trained:
- **Random Forest Regressor** — predict continuous RVSI
- **GBT Regressor** — predict continuous RVSI
- **Random Forest Classifier** — classify instability risk (Low/Medium/High/Critical)
- **GBT Classifier** — classify instability risk

In [36]:
from src.modeling import (
    time_split,
    train_random_forest_regressor,
    train_gbt_regressor,
    train_classification_models,
    evaluate_regression,
    evaluate_classification,
    save_model,
    REGRESSION_FEATURES,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)
print(f"Feature count for modeling: {len(REGRESSION_FEATURES)}")
print(REGRESSION_FEATURES)

Feature count for modeling: 41
['WS50M', 'WS10M', 'ALLSKY_SFC_SW_DWN', 'T2M', 'RH2M', 'PS', 'RHOA', 'CLOUD_AMT', 'QV10M', 'wind_power_index', 'solar_irradiance_index', 'hour', 'month', 'season', 'is_daytime', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'renewable_combined_index', 'temp_humidity_index', 'wind_solar_ratio', 'pressure_density_index', 'wind_direction_variability', 'wind_speed_ratio', 'wind_power_index_lag_1h', 'wind_power_index_lag_6h', 'wind_power_index_lag_12h', 'wind_power_index_lag_24h', 'solar_irradiance_index_lag_1h', 'solar_irradiance_index_lag_6h', 'solar_irradiance_index_lag_12h', 'solar_irradiance_index_lag_24h', 'wind_power_index_rolling_mean_6h', 'wind_power_index_rolling_mean_24h', 'wind_power_index_rolling_std_6h', 'wind_power_index_rolling_std_24h', 'solar_irradiance_index_rolling_mean_6h', 'solar_irradiance_index_rolling_mean_24h', 'solar_irradiance_index_rolling_std_6h', 'solar_irradiance_index_rolling_std_24h']


In [37]:
# Step 6a: Chronological train / test split (80% / 20%)
train_df, test_df = time_split(df_feat, train_ratio=0.8)
print(f"Train rows: {train_df.count():,}")
print(f"Test  rows: {test_df.count():,}")


------------------------------------------------------------
  Chronological Train/Test Split
------------------------------------------------------------



26/05/04 16:11:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  Total rows (after dropping nulls): 83,952


26/05/04 16:11:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  Train: 67,161 rows


...

  Test:  16,791 rows


26/05/04 16:11:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

Train rows: 67,161


26/05/04 16:11:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

Test  rows: 16,791


In [38]:
# Step 6b: Random Forest Regressor
rf_reg_model = train_random_forest_regressor(train_df)
rf_reg_metrics = evaluate_regression(rf_reg_model, test_df, model_name="Random Forest")
print(rf_reg_metrics)


------------------------------------------------------------
  Training Random Forest Regressor
------------------------------------------------------------



26/05/04 16:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:11:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  [OK] Random Forest trained with 100 trees

------------------------------------------------------------
  Evaluating Random Forest (Regression)
------------------------------------------------------------



26/05/04 16:12:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  RMSE: 0.193403
  MAE:  0.143017
  R^2:   0.610913
{'rmse': 0.19340327052837544, 'mae': 0.1430169973507285, 'r2': 0.6109125232057342}


In [39]:
# Step 6c: Gradient Boosted Trees Regressor
gbt_reg_model = train_gbt_regressor(train_df)
gbt_reg_metrics = evaluate_regression(gbt_reg_model, test_df, model_name="GBT")
print(gbt_reg_metrics)


------------------------------------------------------------
  Training Gradient Boosted Trees Regressor
------------------------------------------------------------



26/05/04 16:12:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:12:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  [OK] GBT Regressor trained with 100 iterations

------------------------------------------------------------
  Evaluating GBT (Regression)
------------------------------------------------------------



26/05/04 16:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  RMSE: 0.140728
  MAE:  0.095764
  R^2:   0.793993
{'rmse': 0.14072831882681286, 'mae': 0.09576396090092129, 'r2': 0.7939926934255765}


In [40]:
# Regression comparison bar chart
models_reg = ["Random Forest", "GBT"]
rmse_vals = [rf_reg_metrics["rmse"], gbt_reg_metrics["rmse"]]
mae_vals  = [rf_reg_metrics["mae"],  gbt_reg_metrics["mae"]]
r2_vals   = [rf_reg_metrics["r2"],   gbt_reg_metrics["r2"]]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["steelblue", "coral"]
for ax, metric, vals, label in [
    (axes[0], "RMSE", rmse_vals, "Lower is better"),
    (axes[1], "MAE",  mae_vals,  "Lower is better"),
    (axes[2], "R²",   r2_vals,   "Higher is better"),
]:
    bars = ax.bar(models_reg, vals, color=colors, alpha=0.85)
    ax.set_title(f"Regression {metric}\n({label})", fontweight="bold")
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                f"{v:.4f}", ha="center", va="bottom", fontsize=10)

fig.suptitle("MLlib Regression Model Comparison (RVSI Prediction)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(PLOT_DIR / "model_comparison_regression.png"), dpi=150, bbox_inches="tight")
plt.show()

In [41]:
# Step 6d: Classification models (RF + GBT)
clf_models = train_classification_models(train_df)

rf_clf_metrics  = evaluate_classification(clf_models["rf"],  clf_models["indexer"], test_df, model_name="RF Classifier")
gbt_clf_metrics = evaluate_classification(clf_models["gbt"], clf_models["indexer"], test_df, model_name="GBT Classifier")
print(f"RF  Classifier — Accuracy: {rf_clf_metrics['accuracy']:.4f}  F1: {rf_clf_metrics['f1']:.4f}")
print(f"GBT Classifier — Accuracy: {gbt_clf_metrics['accuracy']:.4f}  F1: {gbt_clf_metrics['f1']:.4f}")


------------------------------------------------------------
  Training Classification Models
------------------------------------------------------------



26/05/04 16:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1


  Training RF Classifier (100 trees, depth 15)...


26/05/04 16:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  [OK] RF Classifier trained -- 100 trees

  Training GBT Classifier (OneVsRest, 100 iters, depth 10)...


...

  [OK] Logistic Regression trained

------------------------------------------------------------
  Evaluating RF Classifier (Classification)
------------------------------------------------------------



26/05/04 16:51:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  Accuracy:  0.6199
  F1 Score:  0.5980
  Precision: 0.6199
  Recall:    0.6199

------------------------------------------------------------
  Evaluating GBT Classifier (Classification)
------------------------------------------------------------



26/05/04 16:51:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 16:51:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 1

  Accuracy:  0.6992
  F1 Score:  0.6948
  Precision: 0.6972
  Recall:    0.6992
RF  Classifier — Accuracy: 0.6199  F1: 0.5980
GBT Classifier — Accuracy: 0.6992  F1: 0.6948


In [60]:
# Classification comparison chart
models_clf = ["RF Classifier", "GBT Classifier"]
acc_vals = [rf_clf_metrics["accuracy"], gbt_clf_metrics["accuracy"]]
f1_vals  = [rf_clf_metrics["f1"],       gbt_clf_metrics["f1"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric, vals in [(axes[0], "Accuracy", acc_vals), (axes[1], "F1 Score", f1_vals)]:
    bars = ax.bar(models_clf, vals, color=["steelblue", "coral"], alpha=0.85)
    ax.set_title(f"Classification {metric}", fontweight="bold")
    ax.set_ylim(0, 1)
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v:.4f}", ha="center", va="bottom", fontsize=10)

fig.suptitle("MLlib Classification Model Comparison (Instability Risk)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(PLOT_DIR / "model_comparison_classification.png"), dpi=150, bbox_inches="tight")
plt.show()

In [43]:
# Step 6e: Save best models
# Pick best regression model by R²
best_reg_model = gbt_reg_model if gbt_reg_metrics["r2"] >= rf_reg_metrics["r2"] else rf_reg_model
best_reg_name  = "gbt_regressor" if gbt_reg_metrics["r2"] >= rf_reg_metrics["r2"] else "rf_regressor"
save_model(best_reg_model, best_reg_name)

# Pick best classifier by F1
best_clf_model = clf_models["gbt"] if gbt_clf_metrics["f1"] >= rf_clf_metrics["f1"] else clf_models["rf"]
best_clf_name  = "gbt_classifier" if gbt_clf_metrics["f1"] >= rf_clf_metrics["f1"] else "rf_classifier"
save_model(best_clf_model, best_clf_name)

print(f"Best regressor  saved: {best_reg_name}")
print(f"Best classifier saved: {best_clf_name}")

  [OK] Model saved to: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/models/gbt_regressor


26/05/04 16:52:39 WARN TaskSetManager: Stage 20343 contains a task of very large size (2014 KiB). The maximum recommended task size is 1000 KiB.
26/05/04 16:52:44 WARN TaskSetManager: Stage 20350 contains a task of very large size (1934 KiB). The maximum recommended task size is 1000 KiB.
26/05/04 16:52:48 WARN TaskSetManager: Stage 20357 contains a task of very large size (2254 KiB). The maximum recommended task size is 1000 KiB.
26/05/04 16:52:52 WARN TaskSetManager: Stage 20364 contains a task of very large size (1852 KiB). The maximum recommended task size is 1000 KiB.


  [OK] Model saved to: /mnt/c/Users/Amer/OneDrive/Desktop/salman/atrifact/models/gbt_classifier
Best regressor  saved: gbt_regressor
Best classifier saved: gbt_classifier



## Section 7: Time Series Analysis (SARIMA)

Classical statistical forecasting of daily RVSI using **SARIMA(1,1,1)(1,1,1,7)** (weekly seasonality).

In [44]:
from src.time_series import run_arima_forecast, _load_daily_rvsi, _adf_test

# Load daily aggregated RVSI from Parquet (uses Spark internally)
daily_rvsi = _load_daily_rvsi()
print(f"Daily RVSI series: {len(daily_rvsi)} days")
print(daily_rvsi.describe())

Daily RVSI series: 3724 days
count    3724.000000
mean        1.101795
std         0.314611
min         0.000000
25%         0.940966
50%         1.107110
75%         1.298872
max         2.124691
Name: rvsi, dtype: float64


In [45]:
# ADF Stationarity Test
is_stationary = _adf_test(daily_rvsi, "Daily RVSI")
print(f"\nSeries is {'stationary' if is_stationary else 'non-stationary'} (p<0.05 threshold)")


  ADF Test -- Daily RVSI
    Statistic: -2.2683
    p-value:   0.1824
    Lags used: 29
    Stationary: No

Series is non-stationary (p<0.05 threshold)


In [46]:
# Fit SARIMA and forecast
sarima_metrics = run_arima_forecast(daily_rvsi, forecast_days=30)
print(f"\nSARIMA Test Metrics:")
print(f"  RMSE: {sarima_metrics['rmse']:.4f}")
print(f"  MAE : {sarima_metrics['mae']:.4f}")
print(f"  R²  : {sarima_metrics['r2']:.4f}")
print(f"  AIC : {sarima_metrics['aic']:.2f}")

------------------------------------------------------------
  SARIMA Forecasting (Daily RVSI)
------------------------------------------------------------
  Total daily observations: 3724

  ADF Test -- Daily RVSI
    Statistic: -2.2683
    p-value:   0.1824
    Lags used: 29
    Stationary: No

  Train: 3664 days, Test: 60 days

  Fitting SARIMA(1,1,1)(1,1,1,7)...
  AIC: -1267.77
  BIC: -1236.76

  SARIMA Forecast Results (60-day test):
    RMSE: 0.1593
    MAE:  0.1228
    R2:   0.0214
  Saved: 15_sarima_forecast.png
  Saved: 16_sarima_diagnostics.png

  30-day future forecast mean RVSI: 0.2713

SARIMA Test Metrics:
  RMSE: 0.1593
  MAE : 0.1228
  R²  : 0.0214
  AIC : -1267.77


In [47]:
# Display SARIMA forecast plot
img = plt.imread(str(PLOT_DIR / "15_sarima_forecast.png"))
plt.figure(figsize=(16, 10)); plt.imshow(img); plt.axis("off")
plt.title("Fig 15: SARIMA Forecast vs Actual Daily RVSI"); plt.tight_layout(); plt.show()

In [48]:
# Display SARIMA diagnostics
img = plt.imread(str(PLOT_DIR / "16_sarima_diagnostics.png"))
plt.figure(figsize=(14, 10)); plt.imshow(img); plt.axis("off")
plt.title("Fig 16: SARIMA Model Diagnostics"); plt.tight_layout(); plt.show()


## Section 8: Deep Learning (TensorFlow / Keras)

Two neural network models:
- **LSTM Regressor** — 24-hour sliding window sequences → RVSI prediction
- **DNN Classifier** — fully connected layers → instability risk (4-class softmax)

In [49]:
from src.deep_learning import (
    train_lstm_regressor,
    train_dnn_classifier,
    run_deep_learning,
    FEATURE_COLS as DL_FEATURE_COLS,
)

print(f"DL features: {len(DL_FEATURE_COLS)}")
print(DL_FEATURE_COLS)

DL features: 25
['WS50M', 'WS10M', 'ALLSKY_SFC_SW_DWN', 'T2M', 'RH2M', 'PS', 'RHOA', 'CLOUD_AMT', 'QV10M', 'wind_power_index', 'solar_irradiance_index', 'hour', 'month', 'season', 'is_daytime', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'renewable_combined_index', 'temp_humidity_index', 'wind_solar_ratio', 'pressure_density_index', 'wind_direction_variability', 'wind_speed_ratio']


In [50]:
# Prepare Pandas DataFrame for DL (drop rows with any missing DL feature)
dl_cols = DL_FEATURE_COLS + ["rvsi", "instability_risk", "timestamp"]
dl_cols_available = [c for c in dl_cols if c in pdf_feat.columns]
pdf_dl = pdf_feat[dl_cols_available].dropna().reset_index(drop=True)
print(f"Rows available for DL: {len(pdf_dl):,}")
print(f"Features used        : {len(DL_FEATURE_COLS)}")

Rows available for DL: 83,975
Features used        : 25


In [51]:
# Step 8a: LSTM Regressor
# Architecture: LSTM(64) → Dropout → LSTM(32) → Dropout → Dense(16) → Dense(1)
# Input: 24-hour sliding window sequences
# Target: RVSI (continuous)

lstm_metrics = train_lstm_regressor(pdf_dl, seq_length=24, epochs=30, batch_size=64)

print(f"\nLSTM Regressor Results:")
print(f"  RMSE : {lstm_metrics['rmse']:.4f}")
print(f"  MAE  : {lstm_metrics['mae']:.4f}")
print(f"  R²   : {lstm_metrics['r2']:.4f}")

------------------------------------------------------------
  Training LSTM Regressor (RVSI Prediction)
------------------------------------------------------------
  Train sequences: (67156, 24, 25)
  Test sequences:  (16771, 24, 25)
Epoch 1/30
892/892 ━━━━━━━━━━━━━━━━━━━━ 26s 25ms/step - loss: 0.0853 - mae: 0.2115 - val_loss: 0.0401 - val_mae: 0.1512
Epoch 2/30
892/892 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - loss: 0.0421 - mae: 0.1520 - val_loss: 0.0351 - val_mae: 0.1366
Epoch 3/30
892/892 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - loss: 0.0331 - mae: 0.1324 - val_loss: 0.0317 - val_mae: 0.1313
Epoch 4/30
892/892 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0274 - mae: 0.1193 - val_loss: 0.0290 - val_mae: 0.1240
Epoch 5/30
892/892 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - loss: 0.0231 - mae: 0.1094 - val_loss: 0.0276 - val_mae: 0.1201
Epoch 6/30
892/892 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - loss: 0.0202 - mae: 0.1017 - val_loss: 0.0264 - val_mae: 0.1175
Epoch 7/30
892/892 ━━━━━━━━━━━━━━━━━━━━ 20s 

In [52]:
# Display LSTM training history + predictions
img = plt.imread(str(PLOT_DIR / "13_lstm_regression.png"))
plt.figure(figsize=(14, 5)); plt.imshow(img); plt.axis("off")
plt.title("Fig 13: LSTM — Training Loss & Prediction Scatter"); plt.tight_layout(); plt.show()

In [53]:
# Step 8b: DNN Classifier
# Architecture: Dense(128) → BN → Dropout → Dense(64) → BN → Dropout → Dense(32) → Dense(4, softmax)
# Target: instability_risk (Low / Medium / High / Critical)

dnn_metrics = train_dnn_classifier(pdf_dl, epochs=30, batch_size=64)

print(f"\nDNN Classifier Results:")
print(f"  Accuracy : {dnn_metrics['accuracy']:.4f}")
print(f"  F1 Score : {dnn_metrics['f1']:.4f}")

------------------------------------------------------------
  Training DNN Classifier (Instability Risk)
------------------------------------------------------------
  Classes: ['Critical', 'High', 'Low', 'Medium']
  Train: 67180, Test: 16795
Epoch 1/30
893/893 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.4209 - loss: 1.2557 - val_accuracy: 0.4554 - val_loss: 1.1537
Epoch 2/30
893/893 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.4615 - loss: 1.1670 - val_accuracy: 0.4620 - val_loss: 1.1389
Epoch 3/30
893/893 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4685 - loss: 1.1524 - val_accuracy: 0.4582 - val_loss: 1.1325
Epoch 4/30
893/893 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4749 - loss: 1.1439 - val_accuracy: 0.4567 - val_loss: 1.1315
Epoch 5/30
893/893 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4754 - loss: 1.1402 - val_accuracy: 0.4628 - val_loss: 1.1265
Epoch 6/30
893/893 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4779 - loss: 1.1368 - val_accuracy: 0.4624 - val_l

In [ ]:
# Display DNN training history
img = plt.imread(str(PLOT_DIR / "14_dnn_classification.png"))
plt.figure(figsize=(14, 5)); plt.imshow(img); plt.axis("off")
plt.title("Fig 14: DNN — Training Loss & Accuracy"); plt.tight_layout(); plt.show() 


## Section 9: Results Summary & Model Comparison

In [55]:
import pandas as pd

# --- Regression Leaderboard ---
reg_leaderboard = pd.DataFrame([
    {"Model": "Random Forest (Spark)",  "RMSE": rf_reg_metrics["rmse"],  "MAE": rf_reg_metrics["mae"],  "R²": rf_reg_metrics["r2"]},
    {"Model": "GBT (Spark)",            "RMSE": gbt_reg_metrics["rmse"], "MAE": gbt_reg_metrics["mae"], "R²": gbt_reg_metrics["r2"]},
    {"Model": "SARIMA (Stats)",         "RMSE": sarima_metrics["rmse"],  "MAE": sarima_metrics["mae"],  "R²": sarima_metrics["r2"]},
    {"Model": "LSTM (Deep Learning)",   "RMSE": lstm_metrics["rmse"],   "MAE": lstm_metrics["mae"],   "R²": lstm_metrics["r2"]},
]).round(4)

reg_leaderboard = reg_leaderboard.sort_values("R²", ascending=False).reset_index(drop=True)
print("=" * 60)
print("   REGRESSION LEADERBOARD (RVSI Prediction)")
print("=" * 60)
print(reg_leaderboard.to_string(index=False))

   REGRESSION LEADERBOARD (RVSI Prediction)
                Model   RMSE    MAE     R²
          GBT (Spark) 0.1407 0.0958 0.7940
 LSTM (Deep Learning) 0.1527 0.1070 0.7572
Random Forest (Spark) 0.1934 0.1430 0.6109
       SARIMA (Stats) 0.1593 0.1228 0.0214


In [56]:
# --- Classification Leaderboard ---
clf_leaderboard = pd.DataFrame([
    {"Model": "RF Classifier (Spark)",  "Accuracy": rf_clf_metrics["accuracy"],  "F1": rf_clf_metrics["f1"]},
    {"Model": "GBT Classifier (Spark)", "Accuracy": gbt_clf_metrics["accuracy"], "F1": gbt_clf_metrics["f1"]},
    {"Model": "DNN Classifier (Keras)", "Accuracy": dnn_metrics["accuracy"],     "F1": dnn_metrics["f1"]},
]).round(4)

clf_leaderboard = clf_leaderboard.sort_values("F1", ascending=False).reset_index(drop=True)
print("=" * 60)
print("   CLASSIFICATION LEADERBOARD (Instability Risk)")
print("=" * 60)
print(clf_leaderboard.to_string(index=False))

   CLASSIFICATION LEADERBOARD (Instability Risk)
                 Model  Accuracy     F1
GBT Classifier (Spark)    0.6992 0.6948
 RF Classifier (Spark)    0.6199 0.5980
DNN Classifier (Keras)    0.4840 0.3869


In [57]:
# Combined comparison chart
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Regression — R²
axes[0].barh(reg_leaderboard["Model"], reg_leaderboard["R²"], color=plt.cm.Blues_r([0.3, 0.5, 0.7, 0.9]))
axes[0].set_xlabel("R² Score (higher = better)", fontsize=11)
axes[0].set_title("Regression R² — All Models", fontsize=13, fontweight="bold")
axes[0].set_xlim(0, 1)
for i, v in enumerate(reg_leaderboard["R²"]):
    axes[0].text(v + 0.005, i, f"{v:.4f}", va="center")

# Classification — F1
axes[1].barh(clf_leaderboard["Model"], clf_leaderboard["F1"], color=plt.cm.Oranges_r([0.3, 0.6, 0.85]))
axes[1].set_xlabel("F1 Score (higher = better)", fontsize=11)
axes[1].set_title("Classification F1 — All Models", fontsize=13, fontweight="bold")
axes[1].set_xlim(0, 1)
for i, v in enumerate(clf_leaderboard["F1"]):
    axes[1].text(v + 0.005, i, f"{v:.4f}", va="center")

fig.suptitle("Model Comparison — Renewable Integration Stability Forecasting",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(str(PLOT_DIR / "final_model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [58]:
# List all generated plots
plots = sorted(PLOT_DIR.glob("*.png"))
print(f"Total plots generated: {len(plots)}")
for p in plots:
    print(f"  {p.name}")

Total plots generated: 19
  01_descriptive_statistics.png
  02_distributions.png
  03_correlation_heatmap.png
  04_time_series_trends.png
  05_hourly_patterns.png
  06_seasonal_boxplots.png
  07_wind_direction.png
  08_missing_data_heatmap.png
  09_seasonal_decomposition.png
  10_acf_pacf.png
  11_rvsi_analysis.png
  12_wpi_vs_sii_scatter.png
  13_lstm_regression.png
  14_dnn_classification.png
  15_sarima_forecast.png
  16_sarima_diagnostics.png
  final_model_comparison.png
  model_comparison_classification.png
  model_comparison_regression.png


In [59]:
# Final print summary
print("\n" + "=" * 65)
print("  PIPELINE COMPLETE")
print("=" * 65)
print(f"  Dataset          : NASA POWER Northern Ireland 2015-2025")
print(f"  Records processed: ~87,660 hourly records")
print(f"  Features engineered: {len(df_feat.columns)} columns")
print(f"  Models trained   : RF, GBT (regression + classification)")
print(f"                     SARIMA, LSTM, DNN")
print(f"  Plots saved      : {len(plots)} PNGs in data/plots/")
print(f"  Models saved     : models/")
print("=" * 65)

# Stop Spark
spark.stop()
print("\nSpark session stopped.")


  PIPELINE COMPLETE
  Dataset          : NASA POWER Northern Ireland 2015-2025
  Records processed: ~87,660 hourly records
  Features engineered: 64 columns
  Models trained   : RF, GBT (regression + classification)
                     SARIMA, LSTM, DNN
  Plots saved      : 19 PNGs in data/plots/
  Models saved     : models/

Spark session stopped.
